# Week 3 · On-Time Classification: precision/recall, threshold, confusion matrix

# Requirements: pip install scikit-learn pandas numpy
# Uses: the `zoro` package (rebuilds features deterministically; no GPU, no API key).

We predict whether a shipment is **late** (the rare, operationally costly class) from
the same features as the ETA model. Because ~80% of shipments are on time, **accuracy is
banned** as a headline metric, a model that always says "on time" would be ~80% accurate
and useless. We report precision/recall/F1 on the *late* class, walk the **threshold
tradeoff**, and read the **confusion matrix**.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # repo root
p = pathlib.Path.cwd()
while not (p / "zoro").is_dir() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))

import numpy as np
import pandas as pd
from zoro import data
print("loaded zoro.data")

### The feature table (same as the ETA notebook)

Identical features, identical split, so the regression and classification results are
comparable. The signal for lateness lives across lane distance, carrier reliability,
weight/value, weather, and calendar features.

In [ ]:
# Build the feature table: join shipments with lane distance and carrier profile,
# impute the planted NaNs (mirrors Week 2 cleaning), and derive calendar features.
def build_features(n=100_000, seed=42):
    ships = data.shipments(n, seed=seed)
    lanes = data.lanes(20, seed=11)
    carriers = data.carriers(20, seed=7)
    df = (ships
          .merge(lanes[["lane_id", "distance_km"]], on="lane_id", how="left")
          .merge(carriers[["carrier_id", "on_time_rate", "fleet_size", "region"]], on="carrier_id", how="left"))
    df["weight_kg"] = df["weight_kg"].fillna(df.groupby("commodity")["weight_kg"].transform("median"))
    df["distance_km"] = df["distance_km"].fillna(df["distance_km"].median())
    df = df.drop_duplicates().reset_index(drop=True)
    df["month"] = df["planned_departure"].dt.month
    df["day_of_week"] = df["planned_departure"].dt.dayofweek
    return df

df = build_features(100_000, seed=42)
print("feature table:", df.shape)

NUM_COLS = ["distance_km", "weight_kg", "value_usd", "on_time_rate", "fleet_size", "month", "day_of_week"]
CAT_COLS = ["weather_severity", "commodity"]

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
prep = ColumnTransformer([
    ("num", StandardScaler(), NUM_COLS),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS),
])
print("features: numeric =", NUM_COLS, "| categorical =", CAT_COLS)

### The time-aware split

Same cut dates as the ETA notebook: train on the earliest months, validate on the next
window, test on the most recent. Never shuffle across time.

In [ ]:
# Time-aware split: chronological, never shuffled. Data spans 2025-01-01 .. 2025-12-30.
CUT1 = "2025-08-01"
CUT2 = "2025-10-01"
train_df = df[df["planned_departure"] < CUT1].reset_index(drop=True)
val_df   = df[(df["planned_departure"] >= CUT1) & (df["planned_departure"] < CUT2)].reset_index(drop=True)
test_df  = df[df["planned_departure"] >= CUT2].reset_index(drop=True)
print("train:", len(train_df), "| val:", len(val_df), "| test:", len(test_df))

### The target: predict the rare class

We flip the label so "positive" means **late**, the class an operations team acts on.
That is where a false alarm (precision) and a miss (recall) both carry real cost, and
where the class imbalance lives.

In [ ]:
train_df["is_late"] = (~train_df["is_on_time"]).astype(int)
val_df["is_late"]   = (~val_df["is_on_time"]).astype(int)
test_df["is_late"]  = (~test_df["is_on_time"]).astype(int)
print("late share - train:", round(float(train_df["is_late"].mean()), 4), "| test:", round(float(test_df["is_late"].mean()), 4))

### Two classifiers

Logistic regression (a linear baseline) and a random forest. Each emits a *probability*
of lateness via `predict_proba`; the label is decided by a threshold. Report
precision/recall/F1 at the default 0.5 threshold first.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_recall_fscore_support

clfs = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=0),
    "RandomForest": RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=0),
}

probas = {}
for name, clf in clfs.items():
    pipe = Pipeline([("prep", prep), ("model", clf)])
    pipe.fit(train_df, train_df["is_late"])
    proba = pipe.predict_proba(val_df)[:, 1]  # P(late)
    probas[name] = (pipe, proba)
    preds = (proba >= 0.5).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(val_df["is_late"], preds, pos_label=1, average="binary", zero_division=0)
    print(f"{name:<18} precision={p:.3f} recall={r:.3f} F1={f1:.3f}  (threshold=0.5, val)")

### The threshold tradeoff

The classifier outputs a probability; *you* choose the threshold that turns it into a
flag. Raising the threshold raises precision (fewer false alarms) and lowers recall
(more misses). The right operating point is where *cost* is lowest, a false alarm costs
a proactive notification, a miss costs a breached SLA. Sweep thresholds and pick the one
that maximizes F1 on validation.

In [ ]:
pipe_lr, proba_lr = probas["LogisticRegression"]

thresholds = np.linspace(0.05, 0.95, 19)
rows = []
for t in thresholds:
    preds = (proba_lr >= t).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(val_df["is_late"], preds, pos_label=1, average="binary", zero_division=0)
    rows.append((round(t, 2), round(p, 3), round(r, 3), round(f1, 3)))

thresh_df = pd.DataFrame(rows, columns=["threshold", "precision", "recall", "f1"])
print(thresh_df.to_string(index=False))
best_row = thresh_df.loc[thresh_df["f1"].idxmax()]
BEST_THRESH = float(best_row["threshold"])
print()
print("best threshold (max F1 on val):", BEST_THRESH)

### The confusion matrix

A confusion matrix is the two-by-two accounting of the classifier's decisions: true
negatives, false positives, false negatives, true positives. Read it at the default 0.5
threshold and again at the chosen threshold, the move you made should be *visible* as a
shift between false alarms and misses.

In [ ]:
from sklearn.metrics import confusion_matrix

proba_test = pipe_lr.predict_proba(test_df)[:, 1]
cm_05   = confusion_matrix(test_df["is_late"], (proba_test >= 0.5).astype(int))
cm_best = confusion_matrix(test_df["is_late"], (proba_test >= BEST_THRESH).astype(int))
print("confusion matrix @0.5          (rows=actual, cols=predicted):")
print(cm_05)
print("confusion matrix @best threshold:")
print(cm_best)

### The metric

Headline number: F1 on the *late* class, evaluated on the held-out test set at the
threshold chosen on validation. This is the number that matters when the classes are
imbalanced.

In [ ]:
preds_test = (proba_test >= BEST_THRESH).astype(int)
p, r, f1, _ = precision_recall_fscore_support(test_df["is_late"], preds_test, pos_label=1, average="binary", zero_division=0)
print("BEST_TEST_F1:", round(float(f1), 3))
print("precision:", round(float(p), 3), "| recall:", round(float(r), 3))